# GCCR Slack Data: Flatten and Anonymize

This notebook reads the raw Slack export (`data/slack-raw-json.zip`), flattens the nested JSON structure to a single table, and anonymizes user identities by replacing real Slack user IDs with sequential anonymous IDs. The output is `data/slack-clean.csv`.

**What this notebook does:**
- Reads `users.json` to build an anonymization map and assign roles (Leadership = workspace admin or owner; Member = all others)
- Reads all channel JSON files and extracts message-level data
- Replaces all real Slack user IDs with anonymous IDs
- Retains message text, channel, date, thread, reactions, and role
- Outputs a flat CSV with no personally identifying information

## Build Anonymization Map

Real Slack user IDs are replaced with sequential anonymous IDs (User_0001, User_0002, etc.) in the order users appear in users.json. Bots and deleted accounts are excluded. Role is assigned based on workspace admin or owner status.

In [1]:
import os
import re
import json
import zipfile
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")
from countrycode import countrycode
import jenkspy

# Move to project root if currently in scripts/
if os.path.basename(os.getcwd()) == "scripts":
    os.chdir("..")

zip_path = os.path.join("data", "slack-raw-json.zip")


In [2]:
# Install required packages if not already installed
import subprocess
subprocess.run(["pip", "install", "pytz", "pandas"], check=True)


CompletedProcess(args=['pip', 'install', 'pytz', 'pandas'], returncode=0)

## Build Anonymization Map

Real Slack user IDs are replaced with sequential anonymous IDs (`User_0001`, `User_0002`, etc.) in the order users appear in `users.json`. Bots and deleted accounts are excluded. Role is assigned based on workspace admin or owner status.

In [3]:
user_map   = {}  # real ID -> anonymous ID
user_roles = {}  # anonymous ID -> role
messages   = []

def anonymize_mentions(text, user_map):
    """Replace <@REAL_ID> mention patterns with <@ANON_ID> using user_map."""
    if not text:
        return text
    def replace(match):
        real_id = match.group(1)
        return f"<@{user_map.get(real_id, real_id)}>"
    return re.sub(r"<@([A-Z0-9]+)>", replace, text)

with zipfile.ZipFile(zip_path, "r") as zf:
    namelist = zf.namelist()

    # Build anonymization map from users.json
    with zf.open("users.json") as f:
        users = json.load(f)
        counter = 1
        for user in users:
            if user.get("is_bot", False) or user.get("deleted", False):
                continue
            real_id  = user["id"]
            anon_id  = f"User_{counter:04d}"
            is_admin = user.get("is_admin", False)
            is_owner = user.get("is_owner", False) or user.get("is_primary_owner", False)
            user_map[real_id]   = anon_id
            user_roles[anon_id] = "Leadership" if (is_admin or is_owner) else "Member"
            counter += 1

    print(f"Users mapped:      {len(user_map):,}")
    print(f"Leadership:        {sum(1 for r in user_roles.values() if r == 'Leadership')}")
    print(f"Members:           {sum(1 for r in user_roles.values() if r == 'Member')}")

    # Flatten messages from all channel JSON files
    channel_files = [
        n for n in namelist
        if n.endswith(".json")
        and n != "users.json"
        and n != "channels.json"
        and n != "integration_logs.json"
    ]

    for filepath in channel_files:
        channel_name = filepath.split("/")[0] if "/" in filepath else filepath.replace(".json", "")
        try:
            with zf.open(filepath) as f:
                for msg in json.load(f):
                    real_id  = msg.get("user", "")
                    anon_id  = user_map.get(real_id)
                    if not anon_id:
                        continue
                    ts = float(msg.get("ts", 0))
                    messages.append({
                        "id":          anon_id,
                        "role":         user_roles[anon_id],
                        "channel":      channel_name,
                        "text":         anonymize_mentions(msg.get("text", ""), user_map),
                        "date":         datetime.fromtimestamp(ts) if ts > 0 else None,
                        "ts":           ts,
                        "thread_ts":    msg.get("thread_ts"),
                        "subtype":      msg.get("subtype"),
                        "reactions":    json.dumps(msg.get("reactions", [])),
                        "word_count":   len(msg.get("text", "").split()),
                        "has_mention":  "@" in msg.get("text", ""),
                        "has_url":      "http" in msg.get("text", "").lower()
                    })
        except Exception:
            continue

df = pd.DataFrame(messages)

print('---')
print(f'Total rows:        {len(df):,}')
print(f'Unique users:      {df["id"].nunique():,}')
print(f'Unique channels:   {df["channel"].nunique():,}')
print(f'Date range:        {df["date"].min()} to {df["date"].max()}')


Users mapped:      660
Leadership:        13
Members:           647
---
Total rows:        22,309
Unique users:      600
Unique channels:   148
Date range:        2020-03-21 19:39:25.000200 to 2022-01-12 16:57:59.004700


## Flatten Messages

All channel JSON files are read and flattened to a single table. Each row is one message. Real user IDs are replaced with anonymous IDs at this stage. Messages from users not in `users.json` (bots, deleted accounts, unknown) are excluded.

## Add Country from Membership

Country is derived by joining Slack users to the membership file on name. The membership file is not shared in the repository as it contains identifying information. Non-matching users are flagged for review.

In [4]:
import unicodedata
import pytz

# Load membership file
membership = pd.read_csv("data/membership-raw.csv", encoding="latin1")

# Standardize country names in membership CSV to resolve variant spellings
# that would otherwise produce duplicate rows in downstream analysis
country_standardize = {
    "Britain (UK)":  "United Kingdom",
    "Korea (South)": "South Korea",
    "Korea, South":  "South Korea",
}
membership["Country of Institution"] = membership["Country of Institution"].replace(country_standardize)
print("Country names standardized.")
print(f"Unique countries in membership: {membership['Country of Institution'].nunique()}")

def strip_accents(text):
    """Normalize unicode and strip accent characters."""
    return "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )

# Common credentials to strip
credentials = [
    r"\bphd\b", r"\bmd\b", r"\bcrnp\b", r"\bdr\.?\b", r"\bprof\.?\b",
    r"\bms\b", r"\bba\b", r"\bdvm\b", r"\brnc\b", r"\bdsc\b",
    r"\bmba\b", r"\bbs\b", r"\bdo\b"
]

def clean_name(name, strip_acc=False):
    """Clean name for matching."""
    if not name:
        return ""
    n = name.lower().strip()
    if strip_acc:
        n = strip_accents(n)
    n = re.sub(r"\.(?=[a-z])", " ", n)
    for cred in credentials:
        n = re.sub(cred, "", n, flags=re.IGNORECASE)
    n = re.sub(r"\(.*?\)", "", n)
    n = re.sub(r",", "", n)
    n = re.sub(r"\s+", " ", n).strip()
    return n

def tokenset(name):
    """Return set of tokens from a cleaned name, min 2 chars to avoid initials."""
    return {t for t in name.split() if len(t) >= 2}

# Build tz -> country map from pytz
tz_country_map = {}
for country_code, tzlist in pytz.country_timezones.items():
    try:
        country_name = pytz.country_names[country_code]
    except KeyError:
        continue
    for tz in tzlist:
        tz_country_map[tz] = country_name

# Add deprecated timezone strings not in pytz.country_timezones
deprecated_tz_map = {
    'Asia/Istanbul':        'Turkey',
    'America/Buenos_Aires': 'Argentina',
    'Asia/Chongqing':       'China',
    'Asia/Calcutta':        'India',
    'Asia/Katmandu':        'Nepal',
    'Asia/Rangoon':         'Myanmar',
    'America/Godthab':      'Greenland',
}
tz_country_map.update(deprecated_tz_map)

# Standardize tz_country_map to match membership standardization
tz_standardize = {
    "Korea, South": "South Korea",
}
tz_country_map = {k: tz_standardize.get(v, v) for k, v in tz_country_map.items()}

# Build user tz map from users.json
user_tz_map = {}  # anon_id -> tz country
for user in users:
    if user.get("is_bot", False) or user.get("deleted", False):
        continue
    real_id = user["id"]
    anon_id = user_map.get(real_id)
    if not anon_id:
        continue
    tz = user.get("tz", "")
    if tz in tz_country_map:
        user_tz_map[anon_id] = tz_country_map[tz]

# Build four name lookup maps: accented and stripped, full and last name
membership["name_acc"]   = membership["Full name"].apply(lambda x: clean_name(str(x), strip_acc=False))
membership["name_strip"] = membership["Full name"].apply(lambda x: clean_name(str(x), strip_acc=True))
membership["last_acc"]   = membership["name_acc"].str.split().str[-1]
membership["last_strip"] = membership["name_strip"].str.split().str[-1]

map_name_acc   = dict(zip(membership["name_acc"],   membership["Country of Institution"]))
map_name_strip = dict(zip(membership["name_strip"], membership["Country of Institution"]))
map_last_acc   = dict(zip(membership["last_acc"],   membership["Country of Institution"]))
map_last_strip = dict(zip(membership["last_strip"], membership["Country of Institution"]))

# Build token set map from membership for fuzzy matching
token_map = []  # list of (token_set, country)
for _, row in membership.iterrows():
    tokens = tokenset(row["name_acc"])
    tokens_strip = tokenset(row["name_strip"])
    if tokens:
        token_map.append((tokens, tokens_strip, row["Country of Institution"]))

def token_match(slack_name_acc, slack_name_strip):
    """Check if all tokens in slack name appear in any membership name."""
    slack_tokens_acc   = tokenset(slack_name_acc)
    slack_tokens_strip = tokenset(slack_name_strip)
    if len(slack_tokens_acc) < 2:
        return None  # need at least 2 tokens to avoid false positives
    for mem_tokens, mem_tokens_strip, country in token_map:
        if slack_tokens_acc.issubset(mem_tokens) or slack_tokens_strip.issubset(mem_tokens_strip):
            return country
    return None

# Match users
id_country_map = {}
no_match = []
match_method = {}

for user in users:
    if user.get("is_bot", False) or user.get("deleted", False):
        continue
    real_id = user["id"]
    anon_id = user_map.get(real_id)
    if not anon_id:
        continue
    raw_name   = user.get("real_name", "")
    c_acc      = clean_name(raw_name, strip_acc=False)
    c_strip    = clean_name(raw_name, strip_acc=True)
    last_acc   = c_acc.split()[-1]   if c_acc.split()   else ""
    last_strip = c_strip.split()[-1] if c_strip.split() else ""

    # Try 1: exact match with accents
    country = map_name_acc.get(c_acc)
    if country:
        id_country_map[anon_id] = country
        match_method[anon_id] = "exact_acc"
        continue

    # Try 2: exact match without accents
    country = map_name_strip.get(c_strip)
    if country:
        id_country_map[anon_id] = country
        match_method[anon_id] = "exact_strip"
        continue

    # Try 3: last name with accents
    country = map_last_acc.get(last_acc)
    if country:
        id_country_map[anon_id] = country
        match_method[anon_id] = "last_acc"
        continue

    # Try 4: last name without accents
    country = map_last_strip.get(last_strip)
    if country:
        id_country_map[anon_id] = country
        match_method[anon_id] = "last_strip"
        continue

    # Try 5: token set intersection
    country = token_match(c_acc, c_strip)
    if country:
        id_country_map[anon_id] = country
        match_method[anon_id] = "token"
        continue

    # Try 6: tz fallback
    country = user_tz_map.get(anon_id)
    if country:
        id_country_map[anon_id] = country
        match_method[anon_id] = "tz"
        continue

    no_match.append((anon_id, raw_name))

# Final safety net: standardize any variant country names that slipped through
final_standardize = {
    "Britain (UK)":  "United Kingdom",
    "Korea (South)": "South Korea",
    "Korea, South":  "South Korea",
}
id_country_map = {k: final_standardize.get(v, v) for k, v in id_country_map.items()}

print(f"Matched (exact, accents):    {sum(1 for v in match_method.values() if v == 'exact_acc')}")
print(f"Matched (exact, stripped):   {sum(1 for v in match_method.values() if v == 'exact_strip')}")
print(f"Matched (last, accents):     {sum(1 for v in match_method.values() if v == 'last_acc')}")
print(f"Matched (last, stripped):    {sum(1 for v in match_method.values() if v == 'last_strip')}")
print(f"Matched (token set):         {sum(1 for v in match_method.values() if v == 'token')}")
print(f"Matched (tz fallback):       {sum(1 for v in match_method.values() if v == 'tz')}")
print(f"No match:                    {len(no_match)}")
print(f"Match rate:                  {len(id_country_map)/660*100:.1f}%")
print()
if no_match:
    print("Still unmatched:")
    for anon_id, raw in no_match:
        print(f"  {anon_id}: {raw!r}")


Country names standardized.
Unique countries in membership: 67
Matched (exact, accents):    408
Matched (exact, stripped):   4
Matched (last, accents):     108
Matched (last, stripped):    0
Matched (token set):         5
Matched (tz fallback):       135
No match:                    0
Match rate:                  100.0%



In [5]:
# Add country column to df
df["country"] = df["id"].map(id_country_map)

print(f'Messages with country:    {df["country"].notna().sum():,}')
print(f'Messages without country: {df["country"].isna().sum():,}')
print(f'Unique countries:         {df["country"].nunique()}')
print()
print("Top 10 countries by message count:")
print(df["country"].value_counts().head(10))


Messages with country:    22,309
Messages without country: 0
Unique countries:         61

Top 10 countries by message count:
country
United States     7540
United Kingdom    2479
Netherlands       2409
Italy             1732
Germany           1268
Turkey            1132
India              792
France             784
Israel             725
Argentina          430
Name: count, dtype: int64


## Anonymize Reactions

User IDs embedded in emoji reaction data are replaced with anonymous IDs using the same user_map.

In [6]:
def anonymize_reactions(reactions_json, user_map):
    """Replace real user IDs in reaction user lists with anonymous IDs."""
    reactions = json.loads(reactions_json)
    for reaction in reactions:
        reaction["users"] = [user_map.get(uid, uid) for uid in reaction.get("users", [])]
    return json.dumps(reactions)

df["reactions"] = df["reactions"].apply(lambda r: anonymize_reactions(r, user_map))
print("Reactions anonymized.")


Reactions anonymized.


## Diagnostics

Verify that anonymization, role assignment, mention replacement, reaction anonymization, country assignment, and thread data are all correct before saving.

In [7]:
# Role distribution and ID format
print("Role distribution:")
print(df["role"].value_counts())
print()
print("Sample IDs (should be User_XXXX format):")
print(df["id"].head())
print()

# Mention anonymization
mentions = df[df["has_mention"] == True]["text"].head(5)
print("Sample messages with mentions (should show User_XXXX not Slack IDs):")
for m in mentions:
    print(f"  {m[:100]}")
print()

# Reaction anonymization
has_reactions = df[df["reactions"] != "[]"]["reactions"].head(3)
print("Sample reactions (users should be User_XXXX format):")
for r in has_reactions:
    print(f"  {json.loads(r)[0]}")
print()

# Thread data
print(f"Messages with thread_ts:    {df['thread_ts'].notna().sum():,}")
print(f"Messages without thread_ts: {df['thread_ts'].isna().sum():,}")
print()

# Country coverage
print(f"Messages with country:    {df['country'].notna().sum():,}")
print(f"Messages without country: {df['country'].isna().sum():,}")
print(f"Unique countries:         {df['country'].nunique()}")
print()
print("Top 10 countries by message count:")
print(df["country"].value_counts().head(10))


Role distribution:
role
Member        16924
Leadership     5385
Name: count, dtype: int64

Sample IDs (should be User_XXXX format):
0    User_0438
1    User_0002
2    User_0328
3    User_0072
4    User_0179
Name: id, dtype: object

Sample messages with mentions (should show User_XXXX not Slack IDs):
  <@User_0125> <@User_0061> <@User_0179>  Buena idea. Español LatinoAmericano 
  Maybe you can team up with <@User_0283>?
  Dear all, <@User_0125> <@User_0012> <@User_0061> I've modified the previous spanish translation addi
  <@User_0328> <@User_0283> Could you discuss whether you could do the Danish translation together and
  I would like to make Finnish translation with <@User_0122>

Sample reactions (users should be User_XXXX format):
  {'name': '+1', 'users': ['User_0236', 'User_0115'], 'count': 2}
  {'name': '+1', 'users': ['User_0003'], 'count': 1}
  {'name': 'pray', 'users': ['User_0060'], 'count': 1}

Messages with thread_ts:    11,917
Messages without thread_ts: 10,392

Messages w

## Add Inactive Users

All 660 workspace users are included in the output with an `active` flag. Active users posted at least one message; inactive users (lurkers) joined the workspace but never posted. This preserves the full membership picture for downstream geographic analysis.

In [8]:
# Add active flag — tag all 660 users, active = posted at least one message
active_ids = set(df["id"].unique())

# Build rows for inactive users (in user_roles but not in messages)
inactive_rows = []
for anon_id, role in user_roles.items():
    if anon_id not in active_ids:
        inactive_rows.append({
            "id":        anon_id,
            "role":      role,
            "channel":   None,
            "text":      None,
            "date":      None,
            "ts":        None,
            "thread_ts": None,
            "subtype":   None,
            "reactions": "[]",
            "word_count": 0,
            "has_mention": False,
            "has_url":    False,
            "country":   id_country_map.get(anon_id),
            "active":    False
        })

# Tag active users in df
df["active"] = True

# Append inactive users — align dtypes to df before concat to avoid FutureWarning
inactive_df = pd.DataFrame(inactive_rows)
inactive_df = inactive_df.reindex(columns=df.columns)
for col in df.columns:
    if col in inactive_df.columns:
        try:
            inactive_df[col] = inactive_df[col].astype(df[col].dtype)
        except (ValueError, TypeError):
            pass
df = pd.concat([df, inactive_df], ignore_index=True)

print(f'Total rows (messages + inactive users): {len(df):,}')
print(f'Total users:    660')
print(f'Active users:   {df["active"].sum():,}')
print(f'Inactive users: {(~df["active"]).sum():,}')
print()
print("Inactive users by country:")
print(df[~df["active"]].groupby("country").size().sort_values(ascending=False))


Total rows (messages + inactive users): 22,369
Total users:    660
Active users:   22,309
Inactive users: 60

Inactive users by country:
country
United States     17
Netherlands       10
Italy              6
Belgium            4
United Kingdom     4
Russia             3
Finland            3
Sweden             2
Argentina          1
Iran               1
India              1
Morocco            1
Mexico             1
Japan              1
Ireland            1
South Korea        1
Serbia             1
Nigeria            1
Turkey             1
dtype: int64


In [9]:
print(f'Total workspace users: {df["id"].nunique():,}')
print(f'Active users:          {df[df["active"]==True]["id"].nunique():,}')
print(f'Inactive users:        {df[df["active"]==False]["id"].nunique():,}')
print(f'Total messages:        {df[df["active"]==True].shape[0]:,}')
print(f'Countries:             {df["country"].nunique()}')

Total workspace users: 660
Active users:          600
Inactive users:        60
Total messages:        22,309
Countries:             62


## COUNTRY LEVEL STATISTICS

In [10]:
# Active messages only for message counts
active_df = df[df["active"] == True].copy()

country_stats = active_df.groupby("country").agg(
    Total_Messages=("id", "count"),
    Unique_Users=("id", "nunique"),
    Active_Days=("date", "nunique")
).reset_index()

country_stats["Messages_per_User"]    = country_stats["Total_Messages"] / country_stats["Unique_Users"]
country_stats["Messages_per_Day"]     = country_stats["Total_Messages"] / country_stats["Active_Days"]
country_stats["Active_Days_per_User"] = country_stats["Active_Days"]    / country_stats["Unique_Users"]

country_stats = country_stats.sort_values("Unique_Users", ascending=False)

# Filter: >= 5 unique users for statistical reliability and anonymity
country_stats_filtered = country_stats[country_stats["Unique_Users"] >= 5].copy()

print("Top 15 countries by sub-population size:")
print(country_stats.head(15)[["country", "Total_Messages", "Unique_Users", "Messages_per_User"]].to_string(index=False))

Top 15 countries by sub-population size:
       country  Total_Messages  Unique_Users  Messages_per_User
 United States            7540           202          37.326733
   Netherlands            2409            48          50.187500
        France             784            41          19.121951
United Kingdom            2479            34          72.911765
         Italy            1732            32          54.125000
        Canada             229            18          12.722222
       Germany            1268            17          74.588235
       Belgium             244            16          15.250000
         India             792            13          60.923077
         Japan             169            13          13.000000
        Turkey            1132            10         113.200000
        Israel             725            10          72.500000
          Iran              97             9          10.777778
     Australia             211             9          23.444444

## Participation Ratio

Message-based: (messages from country X / total messages) / (total users from country X / all users)

Countries where message share exceeds user share score >1 (High Engagement). Countries where message share is below user share score <1 (Low or Moderate Engagement). Countries with no messages score 0 (No Activity).


In [11]:
# Total users per country (all 660 including inactive) — denominator
total_by_country = df.groupby("country")["id"].nunique().reset_index(name="Total_Users")
total_by_country["Total_Prop"] = total_by_country["Total_Users"] / total_by_country["Total_Users"].sum()

# Messages per country (active users only) — numerator
msg_by_country = active_df.groupby("country").size().reset_index(name="Total_Messages")
msg_by_country["Msg_Prop"] = msg_by_country["Total_Messages"] / msg_by_country["Total_Messages"].sum()

# Join and calculate representation ratio
mapdf = total_by_country.merge(msg_by_country, on="country", how="left")
mapdf["Msg_Prop"]       = mapdf["Msg_Prop"].fillna(0)
mapdf["Total_Messages"] = mapdf["Total_Messages"].fillna(0)
mapdf["Part_Ratio"]      = mapdf["Msg_Prop"] / mapdf["Total_Prop"]

# Add ISO3 codes for map joining
mapdf["ISO3"] = countrycode(mapdf["country"].tolist(), origin="country.name", destination="iso3c")

# Jenks natural breaks on non-zero ratios
nonzero = mapdf[mapdf["Part_Ratio"] > 0]["Part_Ratio"].values
print(f"Non-zero ratios: {len(nonzero)}, Unique: {len(set(nonzero))}")
breaks = jenkspy.jenks_breaks(nonzero.tolist(), n_classes=3)
print("Jenks breaks:", [round(b, 3) for b in breaks])

# Assign categories
def assign_category(ratio):
    if ratio == 0:
        return "No Activity"
    elif ratio <= breaks[1]:
        return "Low Engagement"
    elif ratio <= breaks[2]:
        return "Moderate Engagement"
    else:
        return "High Engagement"

mapdf["Category"] = mapdf["Part_Ratio"].apply(assign_category)
print(mapdf[["country", "Total_Users", "Total_Messages", "Part_Ratio", "Category"]].sort_values("Part_Ratio", ascending=False).to_string(index=False))


Non-zero ratios: 61, Unique: 48
Jenks breaks: [np.float64(0.03), np.float64(0.473), np.float64(1.348), np.float64(3.045)]
                          country  Total_Users  Total_Messages  Part_Ratio            Category
                           Turkey           11          1132.0    3.044511     High Engagement
                        Venezuela            1            76.0    2.248420     High Engagement
                          Germany           17          1268.0    2.206654     High Engagement
                           Israel           10           725.0    2.144874     High Engagement
                            Qatar            2           144.0    2.130082     High Engagement
                            Chile            2           140.0    2.070913     High Engagement
                   United Kingdom           38          2479.0    1.929998     High Engagement
                            India           14           792.0    1.673636     High Engagement
                       

## Pearson Correlation: Sub-Population Size vs Participation Ratio

Tests whether the participation ratio is independent of Slack presence (total users per country). Restricted to countries with non-zero ratios.

In [12]:
from scipy import stats

# Restrict to countries with non-zero ratio (active in public channels)
corr_df = mapdf[mapdf["Part_Ratio"] > 0].copy()
r, p = stats.pearsonr(corr_df["Total_Users"], corr_df["Part_Ratio"])
print(f"Pearson correlation: r = {r:.3f}, p = {p:.3f}")
print(f"n = {len(corr_df)} countries with non-zero ratio")


Pearson correlation: r = 0.188, p = 0.148
n = 61 countries with non-zero ratio


## Save final datasets

Because some countries contain a small number of people the reidentification of individuals is possible. Data used for downstream notebooks remove country specific raw values (diversity) or country as an attribute (analysis). These datasets can then be shared with the notebooks that produce the paper figures and artifacts.

In [13]:
df.to_csv("data/slack-explore.csv", index=False)
print(f'Saved: data/slack-explore.csv ({len(df):,} rows, {df["active"].sum():,} active, {(~df["active"]).sum():,} inactive)')

# Create shared analysis file — drop country and text to protect PII
# text contains real names and is not used in network analysis
df["mentions"] = df["text"].apply(
    lambda t: ",".join(re.findall(r"<@(User_\d+)>", str(t))) if pd.notna(t) else ""
)
slack_analysis = df.drop(columns=["country", "text"]).copy()
slack_analysis.to_csv("data/slack-analysis.csv", index=False)
print(f'Saved: data/slack-analysis.csv ({len(slack_analysis):,} rows)')

# Create shared diversity file — proportional data only, no raw counts
mapdf[["country", "ISO3", "Part_Ratio", "Category"]].to_csv("data/slack-diversity.csv", index=False)
print(f'Saved: data/slack-diversity.csv ({len(mapdf)} countries)')

Saved: data/slack-explore.csv (22,369 rows, 22,309 active, 60 inactive)
Saved: data/slack-analysis.csv (22,369 rows)
Saved: data/slack-diversity.csv (62 countries)
